<a href="https://colab.research.google.com/github/karanbisht-kb/AIML/blob/main/GoldAss-ImageNet-pretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [86]:
!find /content/dataset -type d -name ".ipynb_checkpoints" -exec rm -r {} +

In [87]:

import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)
    for d in dirs:
        print("  DIR:", d)
    for f in files[:3]:
        print("  FILE:", f)


/content/dataset
  DIR: train
  DIR: val
  DIR: check
/content/dataset/train
  DIR: truck
  DIR: pedestrian
  DIR: bicycle
  DIR: car
/content/dataset/train/truck
  FILE: t2.jpg
  FILE: t1.jpg
  FILE: img6.jpg
/content/dataset/train/pedestrian
  FILE: p2.jpg
  FILE: img7.jpg
  FILE: p1.jpg
/content/dataset/train/bicycle
  FILE: img3.jpg
  FILE: b2.jpg
  FILE: img4.jpg
/content/dataset/train/car
  FILE: img1.jpg
  FILE: c1.jpg
  FILE: c2.jpg
/content/dataset/val
  DIR: truck
  DIR: pedestrian
  DIR: bicycle
  DIR: car
/content/dataset/val/truck
  FILE: vtimg.jpg
/content/dataset/val/pedestrian
  FILE: vpimg.jpg
/content/dataset/val/bicycle
  FILE: vbimg.jpg
/content/dataset/val/car
  FILE: vcimg.jpg
/content/dataset/check
  FILE: check-c.jpg
  FILE: check-t.jpg
  FILE: check-p.jpg


In [88]:

for class_name in os.listdir('/content/dataset/train'):
    path = os.path.join('/content/dataset/train', class_name)
    if os.path.isdir(path):
        print(class_name, "->", len(os.listdir(path)), "files")


truck -> 4 files
pedestrian -> 4 files
bicycle -> 4 files
car -> 4 files


In [89]:

# ImageNet normalization

transform = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

data_dir = "/content/dataset"  # change path




def is_valid_file(x):
    return x.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))

train_dataset = datasets.ImageFolder(
    os.path.join(data_dir, 'train'),
    transform=transform['train'],
    is_valid_file=is_valid_file
)

val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, 'val'),
    transform=transform['val'],
    is_valid_file=is_valid_file
)

#train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform['train'])
#val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['bicycle', 'car', 'pedestrian', 'truck']


In [90]:

model = models.resnet18(pretrained=True)

# Freeze all layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

# Unfreeze last layer
for param in model.fc.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [91]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [92]:

def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()

        train_acc = correct / len(train_dataset)

        # Validation
        model.eval()
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()

        val_acc = val_correct / len(val_dataset)

        print(f"Epoch {epoch+1}/{epochs} "
              f"Loss: {running_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Acc: {val_acc:.4f}")

train_model(model, train_loader, val_loader, epochs=5)


Epoch 1/5 Loss: 1.5354 Train Acc: 0.0625 Val Acc: 0.5000
Epoch 2/5 Loss: 1.2870 Train Acc: 0.2500 Val Acc: 0.5000
Epoch 3/5 Loss: 1.1398 Train Acc: 0.6250 Val Acc: 0.5000
Epoch 4/5 Loss: 0.9880 Train Acc: 0.7500 Val Acc: 0.7500
Epoch 5/5 Loss: 0.8762 Train Acc: 0.8750 Val Acc: 0.7500


In [93]:

from PIL import Image

def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image = transform['val'](image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example
#print(predict_image("/content/dataset/check/check-c.jpg"))
print(predict_image("/content/dataset/check/check-p.jpg"))
#print(predict_image("/content/dataset/check/check-t.jpg"))
#print(predict_image("/content/dataset/check/check-b.jpg"))



pedestrian
